In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

In [19]:
# ── Constants ─────────────────────────────────────────────────────────────────
DATA_ROOT_PATH    = "/home/frankie/WCIS/FLASH-ADC-CHARACTERIZATION/FINAL_DYNAMIC_CHARACTERISTICS/TOP"
POWER_MODES       = ["HPM", "RPM", "LPM"]
FS_STR_LIST       = ["500k", "1M", "2M", "3M", "4M", "5M", "6M"]
FS_LIST           = [500e3, 1e6, 2e6, 3e6, 4e6, 5e6, 6e6]
FI_LIST           = [99.915e3, 99.976e3, 99.854e3, 99.976e3, 100.098e3, 99.487e3, 98.877e3]
J_LIST            = [1637, 819, 409, 273, 205, 163, 135]
BUFFER_SIZE       = 8192
REF_HI            = 2.8
REF_LO            = 0.8
SHUNT_RESISTANCE  = 1.02

SHUNT_VOLTAGE = {
    "HPM": {"500k": 4.62e-3, "1M": 4.98e-3, "2M": 5.62e-3, "3M": 6.16e-3, "4M": 6.74e-3, "5M": 7.32e-3, "6M": 0},
    "RPM": {"500k": 3.25e-3, "1M": 3.60e-3, "2M": 4.20e-3, "3M": 4.80e-3, "4M": 5.38e-3, "5M": 5.93e-3, "6M": 0},
    "LPM": {"500k": 2.50e-3, "1M": 2.89e-3, "2M": 3.52e-3, "3M": 3.96e-3, "4M": 4.39e-3, "5M": 4.98e-3, "6M": 0},
}

R_LADDER    = 12.2e3   # Ohms, measured resistor ladder per ADC
V_REF_SPAN  = 2.0      # V, REF_HI - REF_LO
P_LADDER    = V_REF_SPAN**2 / R_LADDER   # ~327 uW, same for all modes/freqs

POWER_W = {
    pm: {
        fs: (v**2 / SHUNT_RESISTANCE) / 2 + P_LADDER
        for fs, v in fs_dict.items()
    }
    for pm, fs_dict in SHUNT_VOLTAGE.items()
}

hann_window = np.hanning(BUFFER_SIZE)

In [20]:
# ── Load raw data ─────────────────────────────────────────────────────────────
raw_dfs = {pm: pd.DataFrame() for pm in POWER_MODES}

for power_mode_folder in os.listdir(DATA_ROOT_PATH):
    power_mode_folder_path = os.path.join(DATA_ROOT_PATH, power_mode_folder)
    if os.path.isdir(power_mode_folder_path) and power_mode_folder in POWER_MODES:
        for Fs_folder in os.listdir(power_mode_folder_path):
            Fs_path = os.path.join(power_mode_folder_path, Fs_folder)
            if os.path.isdir(Fs_path):
                for csv_file in glob.glob(os.path.join(Fs_path, "*.csv")):
                    df = pd.read_csv(csv_file)
                    for k, col in enumerate(df.columns):
                        raw_dfs[power_mode_folder][Fs_folder + str(k)] = df[col]


/tmp/ipykernel_95976/2979450427.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  raw_dfs[power_mode_folder][Fs_folder + str(k)] = df[col]


In [21]:
# ── Helper functions ──────────────────────────────────────────────────────────
def alias(harmonic_freq, fs, N):
    a = harmonic_freq % fs
    if a > fs / 2:
        a = fs - a
    return int(round(a / fs * N))

def tabei_ueda(fft_full, fft_mag, N, hann_window):
    """Returns (A, fx, Po) — amplitude (V), fractional bin, phase (rad)."""
    fft_pos = fft_mag[:N//2]
    kmax    = np.argmax(fft_pos[1:]) + 1

    if fft_mag[kmax - 1] > fft_mag[kmax + 1]:
        r  = fft_mag[kmax - 1] / fft_mag[kmax]
        fx = kmax + (1.0 - 2.0*r) / (1.0 + r)
    else:
        r  = fft_mag[kmax + 1] / fft_mag[kmax]
        fx = kmax - (1.0 - 2.0*r) / (1.0 + r)

    dx   = fx - kmax
    dy   = np.pi * dx
    sinc = (dy / np.sin(dy)) if abs(dx) > 1e-10 else 1.0
    CG   = np.sum(hann_window) / N

    A  = (-fft_mag[kmax] / N) * sinc * (dx - 1.0) * (dx + 1.0) / CG * 2.0
    Po = (1 / (2*np.pi)) * np.angle(fft_full[kmax] * np.exp(-1j * dy))
    return A, fx, Po


In [22]:
HIGHLIGHT_FS = "4M"
HIGHLIGHT_PM = "HPM"

RAW_ADC_SIGNAL = []
AC_COUPLED_SIGNAL = []
FFT_OF_UNWINDOWED_SIGNAL = []
HANNING_WINDOW_SIGNAL = []
FFT_OF_HANNING_WINDOW_IN_DB = []
FFT_OF_IDEAL_SIGNAL = []
FFT_OF_NAD = []
FFT_OF_NOISE_ONLY = []

results = []
results_df = {}


for pm in POWER_MODES:
    for fs_str, fs, J in zip(FS_STR_LIST, FS_LIST, J_LIST):

        test_cols = [c for c in raw_dfs[pm].columns if c.startswith(fs_str)]

        # ── Pass 1: accumulate FFT magnitudes ─────────────────────────────
        fft_sum   = None
        valid_cols = []

        for col in test_cols:
            waveform_code = np.array(raw_dfs[pm][col].to_list())
            signal = (waveform_code / 255) * 2.0 - 1.0
            signal = signal - np.mean(signal)
            N = len(signal)

            if np.std(signal) < 0.01:
                print(f"  Skip {pm}/{fs_str}/{col} — flat signal")
                continue

            windowed = signal * hann_window
            fft_full = np.fft.fft(windowed)
            fft_mag  = np.abs(fft_full)

            if fft_sum is None:
                fft_sum = fft_mag.copy()
            else:
                fft_sum += fft_mag

            valid_cols.append(col)

        if not valid_cols:
            print(f"  WARNING: no valid captures for {pm}/{fs_str} — skipping row")
            continue

        # ── Averaged FFT magnitude ─────────────────────────────────────────
        fft_mag_avg = fft_sum / len(valid_cols)

        # Need a complex FFT for T&U phase — use the last capture's fft_full
        # (phase from a single capture; only A and fx matter from the avg)
        waveform_code = np.array(raw_dfs[pm][valid_cols[-1]].to_list())
        signal_last   = (waveform_code / 255) * 2.0 - 1.0
        signal_last   = signal_last - np.mean(signal_last)
        fft_full_last = np.fft.fft(signal_last * hann_window)

        # ── T&U on averaged spectrum ───────────────────────────────────────
        A, fx, Po = tabei_ueda(fft_full_last, fft_mag_avg, N, hann_window)
        f_est     = fx * (fs / N)

        if not np.isfinite(A) or not np.isfinite(f_est) or A <= 0:
            print(f"  WARNING: T&U failed for {pm}/{fs_str}")
            continue

        # ── Pass 2: compute per-capture powers using shared f_est ─────────
        signal_pow_list = []
        noise_pow_list  = []
        harm_pow_list   = []
        noise_fft_mag   = None

        for col in valid_cols:
            waveform_code = np.array(raw_dfs[pm][col].to_list())
            signal = (waveform_code / 255) * 2.0 - 1.0
            signal = signal - np.mean(signal)

            windowed = signal * hann_window
            fft_full = np.fft.fft(windowed)
            fft_mag  = np.abs(fft_full)

            # Re-estimate phase per capture, but use shared A and f_est
            _, _, Po_cap = tabei_ueda(fft_full, fft_mag, N, hann_window)

            t     = np.arange(N) / fs
            ideal = A * np.cos(2 * np.pi * f_est * t + 2 * np.pi * Po_cap)
            noise = signal - ideal

            signal_pow_list.append(np.mean(ideal**2))
            noise_pow_list.append(np.mean(noise**2))

            CG            = np.sum(hann_window) / N
            noise_fft_mag = np.abs(np.fft.fft(noise * hann_window)) / N / CG * 2
            harm_pows     = []
            for k in range(2, 11):
                idx = alias(k * f_est, fs, N)
                if 0 < idx < N//2:
                    harm_pows.append(noise_fft_mag[idx]**2 / 2)
            harm_pow_list.append(np.sum(harm_pows))

            # ── Capture plot data ──────────────────────────────────────────
            if pm == HIGHLIGHT_PM and fs_str == HIGHLIGHT_FS and len(RAW_ADC_SIGNAL) == 0:
                f_axis  = np.fft.fftfreq(N, d=1.0/fs)[:N//2]
                CG_plot = np.sum(hann_window) / N

                fft_unwindowed    = np.abs(np.fft.fft(signal))
                fft_unwindowed_db = 20 * np.log10(fft_unwindowed[:N//2] / np.max(fft_unwindowed[:N//2]) + 1e-12)
                FFT_OF_UNWINDOWED_SIGNAL.extend(fft_unwindowed_db)

                RAW_ADC_SIGNAL.extend(waveform_code)
                AC_COUPLED_SIGNAL.extend(signal)
                HANNING_WINDOW_SIGNAL.extend(windowed)

                fft_db = 20 * np.log10(fft_mag[:N//2] / np.max(fft_mag[:N//2]) + 1e-12)
                FFT_OF_HANNING_WINDOW_IN_DB.extend(fft_db)

                ideal_fft_mag = np.abs(np.fft.fft(ideal * hann_window)) / N / CG_plot * 2
                ideal_fft_db  = 20 * np.log10(ideal_fft_mag[:N//2] / (A / np.sqrt(2)) + 1e-12)
                FFT_OF_IDEAL_SIGNAL.extend(ideal_fft_db)

                nad_fft_mag = np.abs(np.fft.fft(noise * hann_window)) / N / CG_plot * 2
                nad_fft_db  = 20 * np.log10(nad_fft_mag[:N//2] / (A / np.sqrt(2)) + 1e-12)
                FFT_OF_NAD.extend(nad_fft_db)

                noise_only_fft = nad_fft_mag.copy()
                for k in range(2, 11):
                    idx = alias(k * f_est, fs, N)
                    if 0 < idx < N//2:
                        noise_only_fft[max(0, idx-1):idx+2] = 0
                noise_only_db = 20 * np.log10(noise_only_fft[:N//2] / (A / np.sqrt(2)) + 1e-12)
                FFT_OF_NOISE_ONLY.extend(noise_only_db)

        # ── Rest of metrics unchanged ──────────────────────────────────────
        avg_signal_pow = np.mean(signal_pow_list)
        avg_noise_pow  = np.mean(noise_pow_list)
        avg_harm_pow   = np.mean(harm_pow_list)
        

        rms_signal = np.sqrt(avg_signal_pow)
        rms_noise  = np.sqrt(avg_noise_pow)
        rms_harm   = np.sqrt(avg_harm_pow)

        SINAD = 20 * np.log10(rms_signal / rms_noise)
        ENOB  = (SINAD - 1.76) / 6.02

        rms_noise_only = np.sqrt(max(avg_noise_pow - avg_harm_pow, 1e-30))
        SNR  = 20 * np.log10(rms_signal / rms_noise_only)
        THD  = 20 * np.log10(rms_harm / rms_signal) if rms_harm > 0 else -np.inf

        sfdr_mag = noise_fft_mag.copy()
        sfdr_mag[:3] = 0
        SFDR = -20 * np.log10(np.max(sfdr_mag[:N//2]) / (rms_signal * np.sqrt(2)))

        DR   = -20 * np.log10(rms_noise / A)
        
        power = POWER_W[pm][fs_str]
        
        FOMw  = power / (2**ENOB * fs)
        FOMsDR = DR + 10 * np.log10((fs / 2) / (power))
        FOMsSINAD = SINAD + 10 * np.log10((fs / 2) / (power))

        results.append({
            "Power Mode":    pm,
            "Fs":            fs_str,
            "ENOB (bits)":   round(ENOB, 2),
            "SNR (dB)":      round(SNR, 2),
            "SINAD (dB)":    round(SINAD, 2),
            "THD (dBFS)":    round(THD, 2),
            "SFDR (dBFS)":   round(SFDR, 2),
            "DR (dBFS)":     round(DR, 2),
            "Power (W)":     power,
            "FOMw (J/step)": FOMw,
            "FOMs (DR dB)":     FOMsDR,
            "FOMs (SINAD dB)":  FOMsSINAD
        })

        print(f"  {pm}/{fs_str}: ENOB={ENOB:.2f} bits, SINAD={SINAD:.2f} dB, "
              f"SNR={SNR:.2f} dB, THD={THD:.2f} dBFS, captures={len(signal_pow_list)}")


results_df = pd.DataFrame(results, columns=[
    "Power Mode", "Fs", "ENOB (bits)", "SNR (dB)", "SINAD (dB)",
    "THD (dBFS)", "SFDR (dBFS)", "DR (dBFS)", "Power (W)", "FOMw (J/step)", "FOMs (DR dB)", "FOMs (SINAD dB)"
])



  HPM/500k: ENOB=4.71 bits, SINAD=30.13 dB, SNR=36.33 dB, THD=-31.33 dBFS, captures=100
  HPM/1M: ENOB=4.72 bits, SINAD=30.15 dB, SNR=36.09 dB, THD=-31.43 dBFS, captures=100
  HPM/2M: ENOB=4.82 bits, SINAD=30.80 dB, SNR=35.96 dB, THD=-32.37 dBFS, captures=100
  HPM/3M: ENOB=4.53 bits, SINAD=29.03 dB, SNR=35.01 dB, THD=-30.29 dBFS, captures=100
  HPM/4M: ENOB=4.54 bits, SINAD=29.11 dB, SNR=35.12 dB, THD=-30.36 dBFS, captures=100
  HPM/5M: ENOB=0.37 bits, SINAD=3.98 dB, SNR=4.57 dB, THD=-12.90 dBFS, captures=100
  RPM/500k: ENOB=4.83 bits, SINAD=30.85 dB, SNR=36.83 dB, THD=-32.11 dBFS, captures=100
  RPM/1M: ENOB=4.83 bits, SINAD=30.87 dB, SNR=36.65 dB, THD=-32.20 dBFS, captures=100
  RPM/2M: ENOB=4.73 bits, SINAD=30.25 dB, SNR=36.09 dB, THD=-31.56 dBFS, captures=100
  RPM/3M: ENOB=4.55 bits, SINAD=29.12 dB, SNR=35.27 dB, THD=-30.33 dBFS, captures=100
  RPM/4M: ENOB=4.80 bits, SINAD=30.64 dB, SNR=36.47 dB, THD=-31.95 dBFS, captures=100
  RPM/5M: ENOB=0.39 bits, SINAD=4.13 dB, SNR=4.61 dB

In [23]:
top_results_df = results_df

top_results_df

,Power Mode,Fs,ENOB (bits),SNR (dB),SINAD (dB),THD (dBFS),SFDR (dBFS),DR (dBFS),Power (W),FOMw (J/step),FOMs (DR dB),FOMs (SINAD dB)
0,HPM,500k,4.71,36.33,30.13,-31.33,32.82,33.14,0.000338,2.579703e-11,121.829514,118.819206
1,HPM,1M,4.72,36.09,30.15,-31.43,32.97,33.16,0.000340,1.293185e-11,124.839084,121.828777
2,HPM,2M,4.82,35.96,30.80,-32.37,35.06,33.81,0.000343,6.063580e-12,128.449609,125.439312
3,HPM,3M,4.53,35.01,29.03,-30.29,32.21,32.04,0.000346,4.999669e-12,128.403900,125.393597
4,HPM,4M,4.54,35.12,29.11,-30.36,34.65,32.12,0.000350,3.755270e-12,129.686233,126.675935
5,HPM,5M,0.37,4.57,3.98,-12.90,18.37,6.99,0.000354,5.485774e-11,105.477008,102.466706
6,RPM,500k,4.83,36.83,30.85,-32.11,32.74,33.86,0.000333,2.338268e-11,122.614560,119.604265
7,RPM,1M,4.83,36.65,30.87,-32.20,32.89,33.88,0.000334,1.171064e-11,125.625834,122.615533
8,RPM,2M,4.73,36.09,30.25,-31.56,32.74,33.26,0.000337,6.327707e-12,127.991985,124.981689
9,RPM,3M,4.55,35.27,29.12,-30.33,31.90,32.13,0.000339,4.841806e-12,128.589994,125.579698


In [24]:
# ── Constants ─────────────────────────────────────────────────────────────────
DATA_ROOT_PATH    = "/home/frankie/WCIS/FLASH-ADC-CHARACTERIZATION/FINAL_DYNAMIC_CHARACTERISTICS/BOTTOM"

In [25]:
# ── Load raw data ─────────────────────────────────────────────────────────────
raw_dfs = {pm: pd.DataFrame() for pm in POWER_MODES}

for power_mode_folder in os.listdir(DATA_ROOT_PATH):
    power_mode_folder_path = os.path.join(DATA_ROOT_PATH, power_mode_folder)
    if os.path.isdir(power_mode_folder_path) and power_mode_folder in POWER_MODES:
        for Fs_folder in os.listdir(power_mode_folder_path):
            Fs_path = os.path.join(power_mode_folder_path, Fs_folder)
            if os.path.isdir(Fs_path):
                for csv_file in glob.glob(os.path.join(Fs_path, "*.csv")):
                    df = pd.read_csv(csv_file)
                    for k, col in enumerate(df.columns):
                        raw_dfs[power_mode_folder][Fs_folder + str(k)] = df[col]


/tmp/ipykernel_95976/2979450427.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  raw_dfs[power_mode_folder][Fs_folder + str(k)] = df[col]


In [26]:

results = []
results_df = {}


for pm in POWER_MODES:
    for fs_str, fs, J in zip(FS_STR_LIST, FS_LIST, J_LIST):

        test_cols = [c for c in raw_dfs[pm].columns if c.startswith(fs_str)]

        # ── Pass 1: accumulate FFT magnitudes ─────────────────────────────
        fft_sum   = None
        valid_cols = []

        for col in test_cols:
            waveform_code = np.array(raw_dfs[pm][col].to_list())
            signal = (waveform_code / 255) * 2.0 - 1.0
            signal = signal - np.mean(signal)
            N = len(signal)

            if np.std(signal) < 0.01:
                print(f"  Skip {pm}/{fs_str}/{col} — flat signal")
                continue

            windowed = signal * hann_window
            fft_full = np.fft.fft(windowed)
            fft_mag  = np.abs(fft_full)

            if fft_sum is None:
                fft_sum = fft_mag.copy()
            else:
                fft_sum += fft_mag

            valid_cols.append(col)

        if not valid_cols:
            print(f"  WARNING: no valid captures for {pm}/{fs_str} — skipping row")
            continue

        # ── Averaged FFT magnitude ─────────────────────────────────────────
        fft_mag_avg = fft_sum / len(valid_cols)

        # Need a complex FFT for T&U phase — use the last capture's fft_full
        # (phase from a single capture; only A and fx matter from the avg)
        waveform_code = np.array(raw_dfs[pm][valid_cols[-1]].to_list())
        signal_last   = (waveform_code / 255) * 2.0 - 1.0
        signal_last   = signal_last - np.mean(signal_last)
        fft_full_last = np.fft.fft(signal_last * hann_window)

        # ── T&U on averaged spectrum ───────────────────────────────────────
        A, fx, Po = tabei_ueda(fft_full_last, fft_mag_avg, N, hann_window)
        f_est     = fx * (fs / N)

        if not np.isfinite(A) or not np.isfinite(f_est) or A <= 0:
            print(f"  WARNING: T&U failed for {pm}/{fs_str}")
            continue

        # ── Pass 2: compute per-capture powers using shared f_est ─────────
        signal_pow_list = []
        noise_pow_list  = []
        harm_pow_list   = []
        noise_fft_mag   = None

        for col in valid_cols:
            waveform_code = np.array(raw_dfs[pm][col].to_list())
            signal = (waveform_code / 255) * 2.0 - 1.0
            signal = signal - np.mean(signal)

            windowed = signal * hann_window
            fft_full = np.fft.fft(windowed)
            fft_mag  = np.abs(fft_full)

            # Re-estimate phase per capture, but use shared A and f_est
            _, _, Po_cap = tabei_ueda(fft_full, fft_mag, N, hann_window)

            t     = np.arange(N) / fs
            ideal = A * np.cos(2 * np.pi * f_est * t + 2 * np.pi * Po_cap)
            noise = signal - ideal

            signal_pow_list.append(np.mean(ideal**2))
            noise_pow_list.append(np.mean(noise**2))

            CG            = np.sum(hann_window) / N
            noise_fft_mag = np.abs(np.fft.fft(noise * hann_window)) / N / CG * 2
            harm_pows     = []
            for k in range(2, 11):
                idx = alias(k * f_est, fs, N)
                if 0 < idx < N//2:
                    harm_pows.append(noise_fft_mag[idx]**2 / 2)
            harm_pow_list.append(np.sum(harm_pows))

            # ── Capture plot data ──────────────────────────────────────────
            if pm == HIGHLIGHT_PM and fs_str == HIGHLIGHT_FS and len(RAW_ADC_SIGNAL) == 0:
                f_axis  = np.fft.fftfreq(N, d=1.0/fs)[:N//2]
                CG_plot = np.sum(hann_window) / N

                fft_unwindowed    = np.abs(np.fft.fft(signal))
                fft_unwindowed_db = 20 * np.log10(fft_unwindowed[:N//2] / np.max(fft_unwindowed[:N//2]) + 1e-12)
                FFT_OF_UNWINDOWED_SIGNAL.extend(fft_unwindowed_db)

                RAW_ADC_SIGNAL.extend(waveform_code)
                AC_COUPLED_SIGNAL.extend(signal)
                HANNING_WINDOW_SIGNAL.extend(windowed)

                fft_db = 20 * np.log10(fft_mag[:N//2] / np.max(fft_mag[:N//2]) + 1e-12)
                FFT_OF_HANNING_WINDOW_IN_DB.extend(fft_db)

                ideal_fft_mag = np.abs(np.fft.fft(ideal * hann_window)) / N / CG_plot * 2
                ideal_fft_db  = 20 * np.log10(ideal_fft_mag[:N//2] / (A / np.sqrt(2)) + 1e-12)
                FFT_OF_IDEAL_SIGNAL.extend(ideal_fft_db)

                nad_fft_mag = np.abs(np.fft.fft(noise * hann_window)) / N / CG_plot * 2
                nad_fft_db  = 20 * np.log10(nad_fft_mag[:N//2] / (A / np.sqrt(2)) + 1e-12)
                FFT_OF_NAD.extend(nad_fft_db)

                noise_only_fft = nad_fft_mag.copy()
                for k in range(2, 11):
                    idx = alias(k * f_est, fs, N)
                    if 0 < idx < N//2:
                        noise_only_fft[max(0, idx-1):idx+2] = 0
                noise_only_db = 20 * np.log10(noise_only_fft[:N//2] / (A / np.sqrt(2)) + 1e-12)
                FFT_OF_NOISE_ONLY.extend(noise_only_db)

        # ── Rest of metrics unchanged ──────────────────────────────────────
        avg_signal_pow = np.mean(signal_pow_list)
        avg_noise_pow  = np.mean(noise_pow_list)
        avg_harm_pow   = np.mean(harm_pow_list)
        

        rms_signal = np.sqrt(avg_signal_pow)
        rms_noise  = np.sqrt(avg_noise_pow)
        rms_harm   = np.sqrt(avg_harm_pow)

        SINAD = 20 * np.log10(rms_signal / rms_noise)
        ENOB  = (SINAD - 1.76) / 6.02

        rms_noise_only = np.sqrt(max(avg_noise_pow - avg_harm_pow, 1e-30))
        SNR  = 20 * np.log10(rms_signal / rms_noise_only)
        THD  = 20 * np.log10(rms_harm / rms_signal) if rms_harm > 0 else -np.inf

        sfdr_mag = noise_fft_mag.copy()
        sfdr_mag[:3] = 0
        SFDR = -20 * np.log10(np.max(sfdr_mag[:N//2]) / (rms_signal * np.sqrt(2)))

        DR   = -20 * np.log10(rms_noise / A)
        
        power = POWER_W[pm][fs_str]
        
        FOMw  = power / (2**ENOB * fs)
        FOMsDR = DR + 10 * np.log10((fs / 2) / (power))
        FOMsSINAD = SINAD + 10 * np.log10((fs / 2) / (power))

        results.append({
            "Power Mode":    pm,
            "Fs":            fs_str,
            "ENOB (bits)":   round(ENOB, 2),
            "SNR (dB)":      round(SNR, 2),
            "SINAD (dB)":    round(SINAD, 2),
            "THD (dBFS)":    round(THD, 2),
            "SFDR (dBFS)":   round(SFDR, 2),
            "DR (dBFS)":     round(DR, 2),
            "Power (W)":     power,
            "FOMw (J/step)": FOMw,
            "FOMs (DR dB)":     FOMsDR,
            "FOMs (SINAD dB)":  FOMsSINAD
        })

        print(f"  {pm}/{fs_str}: ENOB={ENOB:.2f} bits, SINAD={SINAD:.2f} dB, "
              f"SNR={SNR:.2f} dB, THD={THD:.2f} dBFS, captures={len(signal_pow_list)}")


results_df = pd.DataFrame(results, columns=[
    "Power Mode", "Fs", "ENOB (bits)", "SNR (dB)", "SINAD (dB)",
    "THD (dBFS)", "SFDR (dBFS)", "DR (dBFS)", "Power (W)", "FOMw (J/step)", "FOMs (DR dB)", "FOMs (SINAD dB)"
])



  HPM/500k: ENOB=4.81 bits, SINAD=30.69 dB, SNR=37.14 dB, THD=-31.81 dBFS, captures=100
  HPM/1M: ENOB=4.79 bits, SINAD=30.61 dB, SNR=36.88 dB, THD=-31.78 dBFS, captures=100
  HPM/2M: ENOB=4.93 bits, SINAD=31.43 dB, SNR=37.29 dB, THD=-32.73 dBFS, captures=100
  HPM/3M: ENOB=4.59 bits, SINAD=29.40 dB, SNR=35.75 dB, THD=-30.54 dBFS, captures=100
  HPM/4M: ENOB=4.60 bits, SINAD=29.48 dB, SNR=35.16 dB, THD=-30.85 dBFS, captures=100
  HPM/5M: ENOB=-0.16 bits, SINAD=0.79 dB, SNR=0.94 dB, THD=-15.54 dBFS, captures=100
  RPM/500k: ENOB=5.00 bits, SINAD=31.84 dB, SNR=38.11 dB, THD=-33.01 dBFS, captures=100
  RPM/1M: ENOB=5.00 bits, SINAD=31.88 dB, SNR=38.25 dB, THD=-33.01 dBFS, captures=100
  RPM/2M: ENOB=4.87 bits, SINAD=31.10 dB, SNR=37.74 dB, THD=-32.16 dBFS, captures=100
  RPM/3M: ENOB=4.71 bits, SINAD=30.11 dB, SNR=36.22 dB, THD=-31.32 dBFS, captures=100
  RPM/4M: ENOB=4.94 bits, SINAD=31.52 dB, SNR=37.17 dB, THD=-32.90 dBFS, captures=100
  RPM/5M: ENOB=-0.22 bits, SINAD=0.46 dB, SNR=0.66 

In [27]:
top_results_df

,Power Mode,Fs,ENOB (bits),SNR (dB),SINAD (dB),THD (dBFS),SFDR (dBFS),DR (dBFS),Power (W),FOMw (J/step),FOMs (DR dB),FOMs (SINAD dB)
0,HPM,500k,4.71,36.33,30.13,-31.33,32.82,33.14,0.000338,2.579703e-11,121.829514,118.819206
1,HPM,1M,4.72,36.09,30.15,-31.43,32.97,33.16,0.000340,1.293185e-11,124.839084,121.828777
2,HPM,2M,4.82,35.96,30.80,-32.37,35.06,33.81,0.000343,6.063580e-12,128.449609,125.439312
3,HPM,3M,4.53,35.01,29.03,-30.29,32.21,32.04,0.000346,4.999669e-12,128.403900,125.393597
4,HPM,4M,4.54,35.12,29.11,-30.36,34.65,32.12,0.000350,3.755270e-12,129.686233,126.675935
5,HPM,5M,0.37,4.57,3.98,-12.90,18.37,6.99,0.000354,5.485774e-11,105.477008,102.466706
6,RPM,500k,4.83,36.83,30.85,-32.11,32.74,33.86,0.000333,2.338268e-11,122.614560,119.604265
7,RPM,1M,4.83,36.65,30.87,-32.20,32.89,33.88,0.000334,1.171064e-11,125.625834,122.615533
8,RPM,2M,4.73,36.09,30.25,-31.56,32.74,33.26,0.000337,6.327707e-12,127.991985,124.981689
9,RPM,3M,4.55,35.27,29.12,-30.33,31.90,32.13,0.000339,4.841806e-12,128.589994,125.579698


In [28]:
bottom_results_df = results_df

bottom_results_df

,Power Mode,Fs,ENOB (bits),SNR (dB),SINAD (dB),THD (dBFS),SFDR (dBFS),DR (dBFS),Power (W),FOMw (J/step),FOMs (DR dB),FOMs (SINAD dB)
0,HPM,500k,4.81,37.14,30.69,-31.81,33.72,33.70,0.000338,2.419199e-11,122.387425,119.377111
1,HPM,1M,4.79,36.88,30.61,-31.78,33.75,33.62,0.000340,1.227298e-11,125.293247,122.282944
2,HPM,2M,4.93,37.29,31.43,-32.73,36.13,34.44,0.000343,5.639699e-12,129.079017,126.068714
3,HPM,3M,4.59,35.75,29.40,-30.54,32.92,32.41,0.000346,4.791382e-12,128.773476,125.763169
4,HPM,4M,4.60,35.16,29.48,-30.85,35.20,32.49,0.000350,3.599354e-12,130.054536,127.044231
5,HPM,5M,-0.16,0.94,0.79,-15.54,16.58,3.80,0.000354,7.915391e-11,102.292633,99.282335
6,RPM,500k,5.00,38.11,31.84,-33.01,33.59,34.85,0.000333,2.086691e-11,123.603175,120.592891
7,RPM,1M,5.00,38.25,31.88,-33.01,33.83,34.89,0.000334,1.042418e-11,126.636510,123.626208
8,RPM,2M,4.87,37.74,31.10,-32.16,33.22,34.11,0.000337,5.740209e-12,128.838275,125.827978
9,RPM,3M,4.71,36.22,30.11,-31.32,33.04,33.12,0.000339,4.323600e-12,129.573145,126.562837
